# Import Library

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import random
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input
import xgboost as xgb
from tqdm import tqdm
!pip install gradio --quiet
import gc
import time

# Load Dataset

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    print(root)

In [ ]:
DATA_DIR = "/kaggle/input/datasets/andrialaras/dataset-semua/dataset"

paths = []
labels = []

for cls in os.listdir(DATA_DIR):
    cls_path = os.path.join(DATA_DIR, cls)
    if os.path.isdir(cls_path):
        for img_name in os.listdir(cls_path):
            paths.append(os.path.join(cls_path, img_name))
            labels.append(cls)

df = pd.DataFrame({"path": paths, "label": labels})


# EDA

In [ ]:
# EDA 1 & 2: Validasi jumlah data per kelas dan label
print("Jumlah data per kelas:")
for label in df['label'].unique():
    total = df[df['label'] == label].shape[0]
    print(f"Kelas {label}: {total} citra")

# Mapping nama kelas untuk ditampilkan di grafik
label_mapping = {
    'cyst'  : 'cyst',
    'normal': 'normal',
    'stone' : 'stone',
    'tumor' : 'tumor'
}

df['label_display'] = df['label'].map(label_mapping)

# EDA 3: Visualisasi distribusi kelas
plt.figure(figsize=(10, 6))
ax = sns.countplot(
    x='label_display',
    data=df,
    hue='label_display',
    palette='viridis',
    order=['cyst', 'normal', 'stone', 'tumor'],
    legend=False
)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 9),
                textcoords='offset points')

plt.title('Distribusi Jumlah Data Per Kelas')
plt.xlabel('Nama Kelas')
plt.ylabel('Jumlah Citra')
plt.show()
plt.close()

In [ ]:
# Visualisasi sampel citra per kelas
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, label in enumerate(['normal', 'stone', 'tumor', 'cyst']):
    sample_img = df[df['label'] == label].iloc[0]['path']
    img = cv2.imread(sample_img, cv2.IMREAD_GRAYSCALE)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'{label}')
    axes[i].axis('off')
plt.suptitle('Sampel Citra CT Scan per Kelas')
plt.show()

# Persiapan Data

In [ ]:
# Encoding Label
# Label string dikonversi ke integer agar bisa diproses XGBoost

le = LabelEncoder()
y  = le.fit_transform(df["label"])

print("Mapping kelas:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Contoh sebelum :", df["label"].head().tolist())
print("Contoh sesudah :", y[:5])
df["label"].value_counts()


In [ ]:
channel_count = {}

for path in df["path"]:
    img = cv2.imread(path)
    if img is None:
        continue
    shape = img.shape
    ch = 1 if len(shape) == 2 else shape[2]
    if ch not in channel_count:
        channel_count[ch] = 0
    channel_count[ch] += 1

print("Jumlah citra berdasarkan jumlah channel:")
for ch, total in channel_count.items():
    print(f"{ch} channel: {total} citra")

print("\nShape unik yang ditemukan:")
unique_shapes = set([cv2.imread(p).shape for p in df["path"] if cv2.imread(p) is not None])
print(unique_shapes)


# Preprocessing Semua Data (Sebelum Split)

In [ ]:
# Visualisasi preprocessing pada 1 sampel
sample_path = df["path"].iloc[0]
sample_raw  = cv2.imread(sample_path)
sample_raw  = cv2.resize(sample_raw, (224, 224))
sample_rgb  = cv2.cvtColor(sample_raw, cv2.COLOR_BGR2RGB)
sample_norm = preprocess_input(sample_rgb.astype(np.float32))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(sample_rgb)
axes[0].set_title(f"Sebelum Preprocessing\nMin:{sample_rgb.min()}  Max:{sample_rgb.max()}")
axes[0].axis("off")

disp = (sample_norm - sample_norm.min()) / (sample_norm.max() - sample_norm.min())
axes[1].imshow(disp)
axes[1].set_title(f"Sesudah Preprocessing (Normalisasi VGG16)\nMin:{sample_norm.min():.2f}  Max:{sample_norm.max():.2f}")
axes[1].axis("off")

plt.suptitle("Visualisasi Tahap Preprocessing Citra", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
def preprocess_all_images(path_list, label_list, batch_size=32):
    all_images = []
    all_labels = []
    all_paths  = []  
    total      = len(path_list)
    for start in range(0, total, batch_size):
        batch_paths  = path_list[start:start+batch_size]
        batch_labels = label_list[start:start+batch_size]
        for path, lbl in zip(batch_paths, batch_labels):
            img = cv2.imread(path)
            if img is None:
                print(f"  [SKIP] Gagal baca: {path}")
                continue
            img  = cv2.resize(img, (224, 224))
            rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            norm = preprocess_input(rgb.astype(np.float32))
            all_images.append(norm)
            all_labels.append(lbl)
            all_paths.append(path) 
        if (start // batch_size) % 10 == 0:
            print(f"  Diproses: {min(start+batch_size, total)}/{total} gambar", end="\r")
    print(f"  Selesai  : {total}/{total} gambar")
    return np.array(all_images), np.array(all_labels), np.array(all_paths) 

print("Preprocessing seluruh dataset...")
print(f"Total gambar   : {len(df)}")

all_paths_input  = np.array(df["path"].tolist())
all_labels_input = y

X_all_images, y_all, all_paths_valid = preprocess_all_images(all_paths_input, all_labels_input)

print(f"\nShape X_all_images : {X_all_images.shape}")
if len(all_paths_valid) != len(all_paths_input):
    print(f"{len(all_paths_input) - len(all_paths_valid)} gambar gagal dibaca dan di-skip.")

# Split Data (Setelah Preprocessing)

In [ ]:
# Split array gambar ternormalisasi — Train+Val 85% | Test 15%
paths_arr = all_paths_valid

# Split array gambar (untuk val dan test)
X_trainval_images, X_test_images, y_trainval, y_test = train_test_split(
    X_all_images, y_all,
    test_size=0.15,
    random_state=42,
    stratify=y_all
)

# Split path (untuk referensi test_paths di evaluasi)
train_val_paths, test_paths, train_val_y, test_y = train_test_split(
    paths_arr, y_all,
    test_size=0.15,
    random_state=42,
    stratify=y_all
)

# Bebaskan X_all_images dari RAM setelah split
del X_all_images
gc.collect()

print(f"Train+Val images : {X_trainval_images.shape}")
print(f"Test images      : {X_test_images.shape}")
print(f"\nDistribusi kelas di test set:")
for cls_idx, cls_name in enumerate(le.classes_):
    print(f"  {cls_name}: {np.sum(y_test == cls_idx)} citra")

# K-Fold Cross Validation

In [ ]:
kf          = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_splits = list(kf.split(train_val_paths, train_val_y))

print(f"{'Fold':<6} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 35)
for fold, (train_idx, val_idx) in enumerate(fold_splits):
    print(f"Fold {fold+1}  {len(train_idx):>8} {len(val_idx):>8} {len(test_paths):>8}")
print("-" * 35)
print(f"Test set: {len(test_paths)} ({len(test_paths)/(len(train_val_paths)+len(test_paths))*100:.0f}%)")

In [ ]:
# Distribusi kelas per fold, memastikan tiap kelas terwakili seimbang
class_names_split = sorted([cls for cls in os.listdir(DATA_DIR)
                            if os.path.isdir(os.path.join(DATA_DIR, cls))])

print(f"{'':8} {'':12} {'Total':>7}", end="")
for cls in class_names_split:
    print(f" {cls:>8}", end="")
print()
print("-" * (40 + 10 * len(class_names_split)))

for fold, (train_idx, val_idx) in enumerate(fold_splits):
    for split_name, idx in [("Train", train_idx), ("Val", val_idx)]:
        y_split = train_val_y[idx]
        print(f"Fold {fold+1}   {split_name:<12} {len(idx):>7}", end="")
        for c in range(4):
            print(f" {np.sum(y_split == c):>8}", end="")
        print()
    print()

print(f"Test     {'':12} {len(test_y):>7}", end="")
for c in range(4):
    print(f" {np.sum(test_y == c):>8}", end="")
print()


# Load VGG16

In [ ]:
# VGG16 TIDAK dilatih ulang (trainable=False), hanya digunakan sebagai ekstraktor fitur berbasis ImageNet.

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze semua layer VGG16

# GlobalAveragePooling2D: ubah output conv (7x7x512) → vektor 512-dim
x_gap = GlobalAveragePooling2D(name='gap')(base_model.output)

feature_extractor = Model(
    inputs=base_model.input,
    outputs=x_gap,
    name='VGG16_GAP_Extractor'
)
feature_extractor.summary()

In [ ]:
# Fungsi load + preprocessing gambar (resize, BGR→RGB, normalisasi VGG16)
# Digunakan untuk visualisasi sampel saja di sini
# Preprocessing sesungguhnya dilakukan di dalam load_images() setelah split

def load_and_preprocess(img_path):
    raw = cv2.imread(img_path)
    if raw is None:
        return None, None, None
    rgb  = cv2.cvtColor(cv2.resize(raw, (224, 224)), cv2.COLOR_BGR2RGB)
    norm = preprocess_input(rgb.astype(np.float32))
    return raw, rgb, norm


In [ ]:
sample = df["path"].iloc[0]

raw, rgb, norm = load_and_preprocess(sample)

print("Sebelum :", rgb.min(), rgb.max())
print("Sesudah:", norm.min(), norm.max())


In [ ]:
# Eksplorasi nilai  piksel dan mean substraction
# Ambil satu path citra secara random
random_path = random.choice(train_val_paths)

img = cv2.imread(random_path)
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

print(f"Citra yang dipilih: {random_path}")

brightness = rgb[:,:,0]

# Cari piksel yang nilainya antara 200-254 (terang tapi bukan 255)
mask = (brightness >= 200) & (brightness < 255)
positions = np.argwhere(mask)

if len(positions) > 0:
    # Ambil 5 piksel dari posisi yang memenuhi syarat
    selected = positions[:5]
    print("\nTop 5 piksel terang (nilai 200-254):")
    for pos in selected:
        row, col = pos
        print(f"  Posisi ({row},{col}): R={rgb[row,col,0]}, G={rgb[row,col,1]}, B={rgb[row,col,2]}")
        
        # Langsung hitung mean subtraction nya juga
        r_sub = rgb[row,col,0] - 123.68
        g_sub = rgb[row,col,1] - 116.779
        b_sub = rgb[row,col,2] - 103.939
        print(f"  Setelah mean subtraction: R={r_sub:.3f}, G={g_sub:.3f}, B={b_sub:.3f}")
else:
    print("Tidak ada piksel di rentang 200-254, coba jalankan ulang untuk citra berbeda")

In [ ]:
# Ambil 1 path contoh
sample_path = df["path"].iloc[random.randint(0, len(df)-1)]
sample_label = df[df['path'] == sample_path]['label'].values[0]  # ← ambil label

raw, rgb, norm = load_and_preprocess(sample_path)

plt.figure(figsize=(10, 5))

# RAW
plt.subplot(1, 2, 1)
plt.imshow(raw)
plt.title(f"RAW — Kelas: {sample_label}\nMin:{raw.min()}  Max:{raw.max()}")
plt.axis("off")

# NORMALIZED
plt.subplot(1, 2, 2)
plt.imshow((norm - norm.min()) / (norm.max() - norm.min()))
cy, cx = 112, 112
val = norm[cy, cx]
plt.title(
    f"Normalized VGG16\n"
    f"Min:{norm.min():.2f}  Max:{norm.max():.2f}\n"
    f"(B,G,R): [{val[0]:.2f}, {val[1]:.2f}, {val[2]:.2f}]",
    fontsize=8
)
plt.axis("off")

plt.suptitle("Visualisasi Tahap Preprocessing Citra", fontsize=14)
plt.tight_layout()
plt.subplots_adjust(wspace=0.01)
plt.show()

# Ekstraksi Fitur Val dan Test

In [ ]:
# Ekstraksi fitur val dan test dari array gambar yang sudah dipreprocess
# Dilakukan sekali — tidak diulang per fold karena val/test tidak diaugmentasi

print("Ekstraksi fitur val (dari array gambar ternormalisasi)...")
X_trainval = feature_extractor.predict(X_trainval_images, batch_size=32, verbose=1)
print(f"  Shape X_trainval : {X_trainval.shape}")

print("\nEkstraksi fitur test (dari array gambar ternormalisasi)...")
X_test = feature_extractor.predict(X_test_images, batch_size=32, verbose=1)
print(f"  Shape X_test     : {X_test.shape}")

# Bebaskan array gambar dari RAM — tidak dibutuhkan lagi
del X_trainval_images, X_test_images
gc.collect()
print("\nArray gambar dibebaskan dari RAM.")

In [ ]:
# Visualisasi & Simpan Hasil Ekstraksi Fitur VGG16

class_names_vis = sorted([cls for cls in os.listdir(DATA_DIR)
                          if os.path.isdir(os.path.join(DATA_DIR, cls))])

df_features_full = pd.DataFrame(
    X_test,
    columns=[f'Fitur_{i+1}' for i in range(X_test.shape[1])]
)
df_features_full.insert(0, 'label', [class_names_vis[lbl] for lbl in y_test])
df_features_full.insert(0, 'gambar', [f'gambar_{i+1}' for i in range(len(X_test))])

print("Contoh 10 baris pertama, 10 fitur pertama:")
print(df_features_full.iloc[:10, :12].to_string(index=False))
print(f"\nNilai min  : {X_test.min():.4f}")
print(f"Nilai max  : {X_test.max():.4f}")
print(f"Nilai mean : {X_test.mean():.4f}")

plt.figure(figsize=(14, 5))
sns.heatmap(
    X_test[:10, :20],
    cmap='Blues', annot=True, fmt='.2f',
    xticklabels=[f'F{i+1}' for i in range(20)],
    yticklabels=[f'Gambar {i+1}' for i in range(10)],
    linewidths=0.5
)
plt.title('Heatmap Hasil Ekstraksi Fitur VGG16 (Test Set)\n(10 Sampel x 20 Fitur Pertama)', fontweight='bold')
plt.xlabel('Fitur VGG16')
plt.ylabel('Sampel Gambar')
plt.tight_layout()
plt.show()

csv_path = '/kaggle/working/hasil_ekstraksi_fitur_vgg16.csv'
df_features_full.to_csv(csv_path, index=False)
print(f"\nDisimpan ke {csv_path}")
print(f"   Total baris : {len(df_features_full)} gambar")
print(f"   Total kolom : {df_features_full.shape[1]} (gambar + label + 512 fitur)")


# Training XGBoost

In [ ]:
# TRAINING XGBOOST CROSS VALIDATION 
# Alur per fold:
# Train fold → X_trainval[train_idx] (langsung ambil pakai index)
# Val fold   → X_trainval[val_idx] (langsung ambil pakai index)
# Training XGBoost dan catat akurasi
# Semua data sudah diekstrak sebelumnya, tinggal pakai index

fold_results = []
start_cv     = time.time()

for fold, (train_idx, val_idx) in enumerate(fold_splits):
    start_fold = time.time()

    X_train = X_trainval[train_idx]; y_train = y_trainval[train_idx] # setiap fold, train-val diambil lngsung dgn index (train_idx, val_idx)
    X_val = X_trainval[val_idx]; y_val = y_trainval[val_idx]

    model = xgb.XGBClassifier(
        n_estimators=437,
        max_depth=10,
        learning_rate=0.065049,
        subsample=0.799329,
        colsample_bytree=0.578009,
        gamma=0.155995,
        reg_alpha=0.000003,
        reg_lambda=5.399484,
        min_child_weight=7,
        objective        = 'multi:softprob',
        num_class        = 4,
        eval_metric      = ['mlogloss', 'merror'],
        tree_method      = 'hist',
        early_stopping_rounds = 20,
        random_state     = 42,
        verbosity        = 0,
        n_jobs           = -1
    )
    model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)

    fold_train_acc = accuracy_score(y_train, model.predict(X_train))
    fold_val_acc   = accuracy_score(y_val, model.predict(X_val))
    elapsed_fold   = time.time() - start_fold

    fold_results.append({
        'fold'          : fold + 1,
        'train_accuracy': round(fold_train_acc, 6),
        'val_accuracy'  : round(fold_val_acc, 6),
        'waktu_detik'   : round(elapsed_fold, 2),
    })
    print(f"Fold {fold+1} selesai — Train: {fold_train_acc:.4f} | Val: {fold_val_acc:.4f} | Waktu: {elapsed_fold:.1f}s")

elapsed_cv = time.time() - start_cv
df_fold    = pd.DataFrame(fold_results).set_index('fold')
print("\n", df_fold.to_string())
print(f"\nRata-rata Train Acc : {df_fold['train_accuracy'].mean():.6f}")
print(f"Rata-rata Val Acc   : {df_fold['val_accuracy'].mean():.6f}")
print(f"Std Dev Train Acc   : {df_fold['train_accuracy'].std():.6f}")
print(f"Std Dev Val Acc     : {df_fold['val_accuracy'].std():.6f}")
print(f"\nTotal waktu K-Fold  : {elapsed_cv//60:.0f} menit {elapsed_cv%60:.1f} detik")


In [ ]:
# TRAINING FINAL yang dipakai untuk evaluasi akhir
start_final = time.time()

# Pisahkan sebagian train_val khusus untuk early stopping
X_tr, X_es, y_tr, y_es = train_test_split(
    X_trainval, y_trainval,
    test_size=0.15,
    stratify=y_trainval,
    random_state=42
)

model_xgb = xgb.XGBClassifier(
     n_estimators=437,
        max_depth=10,
        learning_rate=0.065049,
        subsample=0.799329,
        colsample_bytree=0.578009,
        gamma=0.155995,
        reg_alpha=0.000003,
        reg_lambda=5.399484,
        min_child_weight=7,
    objective        = 'multi:softprob',
    num_class        = 4,
    eval_metric      = ['mlogloss', 'merror'],
    tree_method      = 'hist',
    early_stopping_rounds = 20,
    random_state     = 42,
    verbosity        = 0,
    n_jobs           = -1
)

start_train = time.time()
model_xgb.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_es, y_es)],  # ← pakai X_es, bukan X_test
    verbose=False
)

elapsed_train = time.time() - start_train
elapsed_final = time.time() - start_final

print("Training final selesai.")
print(f"Train Accuracy : {accuracy_score(y_tr,   model_xgb.predict(X_tr)):.4f}")
print(f"Test  Accuracy : {accuracy_score(y_test,  model_xgb.predict(X_test)):.4f}")
print(f"Waktu training XGBoost : {elapsed_train//60:.0f} menit {elapsed_train%60:.1f} detik")
print(f"Total waktu training final : {elapsed_final//60:.0f} menit {elapsed_final%60:.1f} detik")
print(f"Jumlah pohon optimal : {model_xgb.best_iteration + 1}")
print(f"n_estimators setting : {model_xgb.n_estimators}")

In [ ]:
results = model_xgb.evals_result() # mengambil riwayat nilai metrik (mlgloss dan merror) yg dicatat xgboost setiap round bossting selama training

epochs  = len(results['validation_0']['mlogloss'])
x_axis  = range(epochs)

train_acc_curve = [1 - e for e in results['validation_0']['merror']] # accuracy = 1 - error Contoh: jika merror = 0.05 (5% salah), maka akurasi = 1 - 0.05 = 0.95 (95% benar)
val_acc_curve   = [1 - e for e in results['validation_1']['merror']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafik kiri — Loss
axes[0].plot(x_axis, results['validation_0']['mlogloss'],
             label='Train Loss', color='#1f77b4')
axes[0].plot(x_axis, results['validation_1']['mlogloss'],
             label='Val Loss', color='#ff7f0e')
axes[0].set_title('Train vs Val Loss', fontweight='bold')
axes[0].set_xlabel('Boosting Round')
axes[0].set_ylabel('Log Loss')
axes[0].legend()

# Grafik kanan — Accuracy
axes[1].plot(x_axis, train_acc_curve,
             label='Train Accuracy', color='#1f77b4')
axes[1].plot(x_axis, val_acc_curve,
             label='Val Accuracy', color='#ff7f0e')
axes[1].set_title('Train vs Val Accuracy', fontweight='bold')
axes[1].set_xlabel('Boosting Round')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('Learning Curve — Hybrid VGG16-XGBoost',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluasi model pada data train, val, dan test

# ===== TRAIN =====
y_trainval_pred = model_xgb.predict(X_trainval)
print(" TRAIN ")
print("Accuracy:", accuracy_score(y_trainval, y_trainval_pred))
print(confusion_matrix(y_trainval, y_trainval_pred))
print(classification_report(y_trainval, y_trainval_pred))

# ===== VAL =====
print("\n VALIDASI (Cross Validation) ")
print(f"Rata-rata Val Accuracy : {df_fold['val_accuracy'].mean():.6f}")
print(f"Std Dev Val Accuracy   : {df_fold['val_accuracy'].std():.6f}")
print(df_fold.to_string())

# ===== TEST =====
y_test_pred = model_xgb.predict(X_test)
print("\n TEST ")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

In [ ]:
class_names = sorted([cls for cls in os.listdir(DATA_DIR)
                      if os.path.isdir(os.path.join(DATA_DIR, cls))])

def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.xlabel('Prediksi Model')
    plt.ylabel('Label Sebenarnya')
    plt.title(title)
    plt.show()

# Confusion Matrix TRAIN
plot_confusion_matrix(y_trainval, y_trainval_pred, class_names,
                      title='Confusion Matrix Train')

# Confusion Matrix TEST
plot_confusion_matrix(y_test, y_test_pred, class_names,
                      title='Confusion Matrix Test')


In [ ]:
report = classification_report(
    y_test, y_test_pred,
    target_names=class_names,
    output_dict=True
)

df_report = pd.DataFrame(report).T
df_report = df_report.loc[class_names, ['precision', 'recall', 'f1-score']]

plt.figure(figsize=(8, 5))
sns.heatmap(
    df_report.astype(float),
    annot=True, fmt='.4f',
    cmap='Blues', vmin=0, vmax=1, linewidths=0.5
)
plt.title('Classification Report Hybrid VGG16-XGBoost (Test Set)', fontweight='bold')
plt.ylabel('Kelas')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Identifikasi gambar yang salah diklasifikasi
test_paths_list = list(test_paths)

misclassified = []
for i in range(len(y_test)):
    true_lbl = class_names[y_test[i]]
    pred_lbl = class_names[int(y_test_pred[i])]
    if true_lbl != pred_lbl:
        misclassified.append({
            'file_path'  : test_paths_list[i],
            'file_name'  : os.path.basename(test_paths_list[i]),
            'true_label' : true_lbl,
            'pred_label' : pred_lbl
        })

df_miss = pd.DataFrame(misclassified)
print(f"Total misklasifikasi : {len(df_miss)} dari {len(y_test)} data test")

if len(df_miss) == 0:
    print("Tidak ada misklasifikasi model mengklasifikasikan semua data test dengan benar.")
else:
    print(f"\nJumlah per kombinasi kelas:")
    print(df_miss.groupby(['true_label', 'pred_label']).size().reset_index(name='jumlah').to_string(index=False))
    df_miss.to_csv('/kaggle/working/misclassified_files.csv', index=False)
    print(f"\nDisimpan ke /kaggle/working/misclassified_files.csv")
    print(df_miss.to_string(index=False))


In [ ]:
plt.figure(figsize=(14, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    img = cv2.imread(test_paths_list[i])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.axis('off')
    true_lbl = class_names[y_test[i]]
    pred_lbl = class_names[int(y_test_pred[i])]
    color    = 'green' if true_lbl == pred_lbl else 'red'
    plt.title(f"Asli : {true_lbl}\nPred : {pred_lbl}", fontsize=7, color=color)

plt.suptitle('Prediksi Model Hybrid VGG16-XGBoost (10 Sampel Test)', fontweight='bold')
plt.tight_layout()
plt.show()
